## Load the extension

In [1]:
%load_ext autoreload
%aimport -sparkmagic # it loses the references to the sessions if it reloads
%autoreload 2

In [2]:
%load_ext livy_uploads.magics

## Fetching remote variable

In [3]:
print(list(sorted(globals())))

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
0,None,pyspark,idle,,,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

['HiveContext', 'StreamingContext', '__builtins__', 'cloudpickle', 'sc', 'spark', 'sqlContext']

In [4]:
%%local
from livy_uploads.magics import get_session

session = get_session()
info = session.refresh()
print(info)

initial_id = info['id']

{'id': 0, 'name': None, 'appId': None, 'owner': None, 'proxyUser': None, 'state': 'idle', 'kind': 'pyspark', 'appInfo': {'driverLogUrl': None, 'sparkUiUrl': None}, 'ttl': None, 'driverMemory': '1000M', 'driverCores': 0, 'executorMemory': None, 'executorCores': 2, 'conf': {'spark.ui.enabled': 'true'}, 'archives': [], 'files': [], 'heartbeatTimeoutInSecond': 0, 'jars': [], 'numExecutors': 0, 'pyFiles': [], 'queue': None}


## Sending local variable

In [15]:
%%local

from livy_uploads.executor import LivyExecutorMaster


In [17]:
%%local

url = session.apply(LivyExecutorMaster())
url

'wss://livy:45789'

In [13]:
%%local

url is None

True

In [ ]:
%get_obj_from_spark -n foo_total

In [ ]:
%local

assert foo_total == 9

## Running commands

In [ ]:
%%shell_command

ls -lahF .

In [ ]:
%%shell_command

bash -c 'echo foo && exit 42'

In [ ]:
%%local

assert shell_output == 'foo\n'
assert shell_returncode == 42


## Sending local file

In [ ]:
%local !ls -lahF

In [ ]:
%send_path_to_spark -p magics.ipynb

In [ ]:
%%shell_command

ls -lahF | grep magics

In [ ]:
%%local

assert 'magics.ipynb' in shell_output

## Sending local directory

In [ ]:
%local !find sample-dir/

In [ ]:
%send_path_to_spark -p sample-dir/

In [ ]:
%%shell_command
pwd

In [ ]:
%%shell_command

find "$PWD/sample-dir"

In [ ]:
%%local

assert 'sample-dir/' in shell_output

## Following session logs

In [ ]:
%logs_follow -p 500

In [ ]:
%logs_follow -p 500

In [ ]:
sc._gateway.jvm.java.lang.System.err.println('Hello World')

In [ ]:
%logs_follow -p 500

In [ ]:
%local

assert 'Hello World' in '\n'.join(logs_lines)

## Making sure it kills running sessions

In [ ]:
%%configure -f
{"name": "test-1"}

In [ ]:
%%local
from livy_uploads.magics import get_session

info = get_session().refresh()
print(info)

initial_id = info['id']

In [ ]:
%%configure -f
{"name": "test-1"}

In [ ]:
%%local
from livy_uploads.magics import get_session

info = get_session().refresh()
print(info)

other_id = info['id']

In [ ]:
%%local

assert other_id != initial_id

In [ ]:
%%local

from livy_uploads.magics import get_session
from livy_uploads.session import LivySession

endpoint = get_session()
sessions = list(LivySession.list(endpoint))
sessions

In [ ]:
%%local

assert len(sessions) == 1

session = sessions[0]

assert session.session_name == 'test-1'
assert session.session_id == other_id